In [ ]:
"""
only needed to run once to load the data into the database. saving for shits and gigs

can also now use the rotowire scrape to spot fill if any days are missed during the season
"""
import requests, json
import pandas as pd
import numpy as np

##### importing custom modules from the projects folder
import sys
from pathlib import Path
# Start at current working directory
current = Path.cwd()
# Walk up the tree until config.py is found or root is reached
for parent in [current] + list(current.parents):
    config_path = parent / "config.py"
    if config_path.exists():
        sys.path.append(str(parent))
        import config # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
        break
else:
    raise FileNotFoundError("config.py not found in any parent directories")
import scripts.functions.NBAhelperfunctions as hf

# rotowire historical odds for total and spread. NO MONEYLINE  goes back to 2017-18 season

In [ ]:
# pull team ids from db for mapping
# rotowire uses the same team abbreviations as my db teamAbbr
db_engine = hf.connect_to_database()
with db_engine.connect() as conn:
    teams = pd.read_sql(
        sql = 'SELECT teamAbbr, espnTid FROM teams',
        con= conn
    )
team_dict = dict(zip(teams['teamAbbr'], teams['espnTid']))

# get the data from rotowire website. this url returns the entire history dataset they have back to 2017, 
# up to the last full day of games
headers = {
    'Host': 'www.rotowire.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:144.0) Gecko/20100101 Firefox/144.0',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'DNT': '1',
    'Sec-GPC': '1',
    'Connection': 'keep-alive',
    'Referer': 'https://www.rotowire.com/betting/nba/archive.php',
    'Cookie': 'PHPSESSID=1010a68a7a2ab43c750e881dc4f90c67; cookieyes-consent=consentid:Y20zNVltVERaZ2Z3ZExxaE8yMUVKU2dtUXFHeEZJS3U,consent:yes,action:no,necessary:yes,functional:yes,analytics:yes,performance:yes,advertisement:yes,other:yes; g_uuid=89bb25a1-9acc-45e4-ac16-c051aed33537; cohort_id=3; _ga_DJZM5GNYZ8=GS2.1.s1762610081$o51$g1$t1762610132$j22$l0$h1953729589; _ga=GA1.1.1911159050.1736898965; _au_1d=AU1D-0100-001736898967-KC8Y7NNE-Y6UV; euconsent=CQLOJoAQLOJoAGRABAENBYFgAAAAAAAAAAAAAAAVggAAAAAA.YAAAAAAAAAAA,; cto_bundle=qbIg-V9NN203aUdZJTJCNUx5aUsyRnJpVHc0azREJTJCMkJoWWg1bHozNUFTbnM3UmZqWEVCdHFteWhTcDVkUFJmc24lMkJnOUJZcG9HT1Q1Q3o0MFBqVFlyS09OZXdHenRJOTdBVmRPRkxDQzVRRHdHNnFYMkpET05XUlVkcGJFaWJHT0RpYVFzMEN5SW0xUzVMTFp5Y1dWVFVYYjRQV25UMDh0Y1ZYbyUyRlpNTmhKVjVOVk84cG9hWnNPeDVuWkMlMkIlMkZ4VnZzbGQlMkZiSg; cto_bidid=_3kuBF9OdDhPRVZYMHJ0Mm9IU1Y5NVVVY1FuRTRkOVpCdjlaJTJCZGM3aW91YkU1Y29Ic1N1Nk1xOHJhYnhIMkwlMkJOODJYbXJuNVRTRFVXNFpVMUZIOUhDSm1vZzVybHU3NUNNMHZkN2F0Z2xMeFhFaEklM0Q; _ga_FVWZ0RM4DH=GS2.1.s1761695340$o43$g0$t1761695340$j60$l0$h0; _lc2_fpi=ee48b0c2def8--01jjfeqwe5xs70dp1q6xdmcnzy; _lc2_fpi_meta=%7B%22w%22%3A1737833050565%7D; _cc_id=382d2e2c652690d3af967073410eb735; cto_dna_bundle=t9ehFF9JUURxMUQ2alJidm5yNFRwcFozU0E1UVhUJTJGRFlHNDNjVDJTSUVMelFGUyUyRjBHS0NSSm1mOHZNOXBkTkZBeDQxSGZ2TWhSVUNMZFZiTEJuZEpiWnMyNlElM0QlM0Q; _tt_enable_cookie=1; _ttp=Y7oNZ6GhF7I80if0fXvYesjlkDa.tt.1; 33acrossIdTp=Tg6wSz%2FBcdVJj2gKbImR6hB3Sebjl5qJpX1U5Fkbgws%3D; idw-fe-id=e90a95b4-e86d-4876-805c-ada63bcbcfff; __gads=ID=b282747500baff44:T=1737833053:RT=1761694233:S=ALNI_MaBArO9q9WZiiiL1FbYeDhvAOeHYA; __gpi=UID=00000fee44281a03:T=1737833053:RT=1761694233:S=ALNI_MYO3WueomqgLirGOjpXiboLQctcsQ; uuid=6E0EC57D-03FB-4CA6-8EFF-D00BA6D08015; sharedId=9f81a9e5-b52f-4174-88cc-e8e82cf860fc; sharedId_cst=zix7LPQsHA%3D%3D; _li_ss=CgA; connectId={"ttl":86400000,"lastUsed":1761694230914,"lastSynced":1761694230914}; ttcsid_CRLDHLJC77U51LO9QM5G=1744841529167.4.1744841529509; ttcsid=1744841529168.4.1744841529168; rw_tsd=1762610081__(direct)__(none)__(not set)__web__desktop; _gcl_au=1.1.2118381701.1761694229; _fbp=fb.1.1761694229103.775235795715729040; intercom-id-bfhjit7z=3580d51d-69d9-460f-a2ba-a59615f9dfc8; intercom-session-bfhjit7z=; intercom-device-id-bfhjit7z=c6d236b4-c3cd-49e9-a7aa-ea06df07e055; hb_insticator_uid=bcbb3954-8aa1-429f-8ef1-d8575736994b; __eoi=ID=b97afe3679878ef6:T=1761694233:RT=1761694233:S=AA-AfjYjPDAhTZU4npQaEV6JF2mr; _lr_env_src_ats=false; pbjs-unifiedid=%7B%22TDID%22%3A%22f5814713-9bbf-4568-897b-73224dbc34a3%22%2C%22TDID_LOOKUP%22%3A%22TRUE%22%2C%22TDID_CREATED_AT%22%3A%222025-09-28T23%3A30%3A39%22%7D; pbjs-unifiedid_cst=YiwPLDosoA%3D%3D; pbjs-unifiedid_last=Tue%2C%2028%20Oct%202025%2023%3A30%3A40%20GMT; _vwo_uuid_v2=DE174984BEC99CBF915E4F5C8FAAAC6D9|291ada161bdba5be2614e65ac5aaf624; _vwo_uuid=DE174984BEC99CBF915E4F5C8FAAAC6D9; _vwo_ds=3%3Aa_1%2Ct_1%3A0%241762275972%3A27.35666183%3A%3A%3A3_1%2C2_1%3A0%3A1762275972%3A1762275972; _vis_opt_s=1%7C; __stripe_mid=288c4141-3cfc-423b-a889-c107e6f0a0fe942431; g_device=windows%7Cdesktop; _vis_opt_test_cookie=1; rwlanding=%252Fbetting%252Fnba%252Farchive.php; g_sid=1762610080370.un5zhc9e; _rdt_uuid=1761694228917.08c5a2cc-d0c4-4641-8f17-151c9d049cd5; _uetsid=79d8ada0bcaa11f08a4219c9c3793b79; _uetvid=16bdcb20b45611f0872639b99b04fe34',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'TE': 'trailers'
}

url = 'https://www.rotowire.com/betting/nba/tables/games-archive.php'
r = requests.get(url, headers = headers)

data = json.loads(r.text)
df = pd.DataFrame(data)

data = json.loads(r.text)
df = pd.DataFrame(data)

# -------------------------------------------------------------------
# process data for loading to db
# -------------------------------------------------------------------
# they have a total column in there that is the actual game score total
# i use total as the game line in my db
df = df.drop(['total'], axis=1)

df = df.rename(columns={
    'game_date':'gameDate',
    'game_over_under':'total',
    'line':'homeSpread'
})

df['espnHomeTid'] = df['home_team_stats_id'].map(team_dict)
df['espnAwayTid'] = df['visit_team_stats_id'].map(team_dict)

# rotowire uses the first year of the season to label the season
# most of my data uses the ending year as the season label so I am updating to match
df['season'] = df['season'].astype(int)
df['season'] = df['season'] + 1
df['homeSpread'] = df['homeSpread'] * 1
df['awaySpread'] = df['homeSpread'] * -1
df['total'] = df['total'] * 1

df['gameDate'] = pd.to_datetime(df['gameDate']).dt.date
df['gameDate'] = df['gameDate'].astype(str)

# create 
df['espnGId'] = df['gameDate'].str.replace('-','') + df['espnHomeTid'].astype(str) + df['espnHomeTid'].astype(str)



df['gameDate'] = pd.to_datetime(df['gameDate']).dt.date
# cols to keep
cols = [
    'espnGId', 'gameDate', 'season', 'espnHomeTid', 'espnAwayTid', 'total', 'homeSpread', 'awaySpread'
]
df = df[cols]

In [ ]:
# load the specific dates needed 
loader = df[(df['gameDate'] == pd.to_datetime('2025-11-07').date())]


In [ ]:
# load to db
db_engine = hf.connect_to_database()
with db_engine.connect() as conn:
    loader.to_sql(
        name='gameodds',
        con = conn,
        if_exists='append',
        index=False
    )

# random dataset found with historical spread, money line and total back to 2008 through 1/16/23

In [ ]:
df = pd.read_csv(r"C:\Users\jrbrz\Desktop\projects\projects\propfarm\data\2008_2023_teamGameOdds.csv")
map_tids = {
    'Utah':26, 
    'LA Lakers':13, 
    'Houston':10, 
    'San Antonio':24, 
    'Portland':22, 
    'Golden State':9, 
    'New Jersey':17, 
    'Dallas':6, 
    'Cleveland':5, 
    'Seattle':25, 
    'Washington':27, 
    'Orlando':19, 
    'Chicago':4, 
    'Sacramento':23, 
    'Milwaukee':15, 
    'Toronto':28,
    'New Orleans':3, 
    'Memphis':29, 
    'Denver':7, 
    'Philadelphia':20, 
    'Indiana':11, 
    'Miami':14, 
    'Detroit':8, 
    'Phoenix':21, 
    'New York':18, 
    'Atlanta':1, 
    'Minnesota':16, 
    'LA Clippers':12, 
    'Charlotte':30, 
    'Boston':2, 
    'Oklahoma City':25, 
    'Brooklyn':17
}
df['team'] = df['team'].map(map_tids)
df['opponent'] = df['opponent'].map(map_tids)

df['espnHomeTid'] = np.where(
    df['home/visitor'] == 'vs',
    df['team'],
    df['opponent']
)
df['espnAwayTid'] = np.where(
    df['home/visitor'] == 'vs',
    df['opponent'],
    df['team']
)
df['homeMoneyline'] = np.where(
    df['home/visitor'] == 'vs',
    df['moneyLine'],
    df['opponentMoneyLine']
)
df['awayMoneyline'] = np.where(
    df['home/visitor'] == 'vs',
    df['opponentMoneyLine'],
    df['moneyLine']    
)
df['homeSpread'] = np.where(
    df['espnHomeTid'] == df['team'],
    df['spread'],
    df['spread'] * -1
)
df['awaySpread'] = np.where(
    df['espnHomeTid'] == df['team'],
    df['spread'] * -1,
    df['spread']
)

df = df.rename(columns={
    'date':'gameDate'
})
df['espnGid'] = df['gameDate'].str.replace('-','') + df['espnHomeTid'].astype(str) + df['espnHomeTid'].astype(str)
df.drop_duplicates(subset=['espnGid'], inplace=True)

df = df[[
    'espnGid', 'gameDate', 'season',  'espnHomeTid', 'espnAwayTid', 'total', 'homeMoneyline', 'awayMoneyline', 'homeSpread', 'awaySpread'
]]

df['espnGid'] = df['espnGid'].astype(int)

In [ ]:
"""
df.to_sql(
    name='gameodds', 
    con=config.map_conn_str['nba'], 
    if_exists='append', 
    index=False)
"""